# 17 — What-ifs: policy and stress (Part 3, Stage 7)

**Does:** re-runs the Stage 5 equilibrium (24 strategic firms) under policy changes and stress scenarios, on a 60-day sample spread evenly over spare capacity.

**Runs:** `src/run_part3.py stage7` (~6 min). **Plan:** `docs/part3_plan.md`, Stage 7. **Logged in:** `docs/model_log.md`, Step 27.

## What-ifs (Step 27)

| What-if | Change |
|---|---|
| Base | Stage 5 settings |
| Cap Rs 5,000 | Price cap lowered from Rs 10,000; bids stop at Rs 4,490 (5,000 × 0.9 − 10, the Step 23 loss margin) |
| Contracts 50% | Every firm has sold half of its at-cost output forward at a fixed price (contract for differences), so on that half a higher spot price earns it nothing |
| Coal −10% | Coal availability × 0.9 in every state (coal shortage) |
| Demand +5% | Every state's demand × 1.05 |

Not run (logged as limits): must-offer (Stages 4–7 already bid only, no withholding), splitting the largest firm (the LP keys blocks by state and fuel, so two owners of one block need a new block type), transmission limits (the model has none).

**Hypotheses:** the lower cap only bites on the few cap days; contracts cut markups roughly in half; stress raises market power **more** than it raises the competitive price.

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dispatch import PROCESSED
import run_part3

def load(file, stage):
    """Read a Part 3 result; run its stage first if the file is missing (see src/run_part3.py)."""
    path = PROCESSED / file
    if not path.exists():
        print(f"{file} missing: running {stage} (see src/run_part3.py for run times)")
        run_part3.STAGES[stage]()
    return pd.read_csv(path, parse_dates=["Date"])

structure = pd.read_csv(PROCESSED / "market_structure_daily.csv", parse_dates=["Date"])
national = pd.read_csv(PROCESSED / "market_structure_national_daily.csv", parse_dates=["Date"]).set_index("Date")
crore = 1e7

w = load("strategic_whatif_daily.csv", "stage7")
w["Markup_pct"] = 100 * (w["Price_avg_eq"] / w["Price_avg_cost"] - 1)
w["Extra_consumer_cr_h"] = (w["Consumer_cost_eq"] - w["Consumer_cost_cost"]) / crore
t = w.groupby("What_if", sort=False).agg(
    Competitive_price=("Price_avg_cost", "mean"), Equilibrium_price=("Price_avg_eq", "mean"),
    Avg_markup_pct=("Markup_pct", "mean"), Max_markup_pct=("Markup_pct", "max"),
    Days_markup_over_5pct=("Markup_pct", lambda m: int((m > 5).sum())),
    Extra_consumer_cr_h=("Extra_consumer_cr_h", "mean"), Unserved_MW_max=("Unserved_MW_eq", "max"),
    Converged=("Status", lambda s: int((s == "converged").sum())))
base = t.loc["Base"]
t["Competitive_price_vs_base_%"] = 100 * (t["Competitive_price"] / base["Competitive_price"] - 1)
t["Equilibrium_price_vs_base_%"] = 100 * (t["Equilibrium_price"] / base["Equilibrium_price"] - 1)
t.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = t.index.tolist()
ax.bar(order, t["Avg_markup_pct"], color=["tab:gray"] + ["tab:blue"] * 2 + ["tab:red"] * 2)
ax.set_ylabel("Average equilibrium markup (%)")
ax.set_title("Market power under each what-if (60-day sample)")
plt.tight_layout(); plt.show()

## Stage 7 conclusions

_(filled in after the run)_